# Hand2Robot 2.0 — old vs new, on Colab, with videos

Runs the pipeline on **our** clips three ways and saves everything — numbers **and videos** — to Google Drive. Nothing here touches the live servers.

| | hand | distances |
|---|---|---|
| **A — today** | MediaPipe | guessed lens, table from the hand |
| **B** | MediaPipe | MoGe-2 depth model: real lens, table, hand size |
| **C** | WiLoR | MoGe-2 (only when WiLoR is switched on, cell 1) |

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Put the clips in Drive: **My Drive / hand2robot2 / clips /** (`.mp4`; with its `.mp4.camera.json` beside it if the new app recorded it). Optional `hand2robot2/truth.csv`: `clip,object_width_cm`.

**Output, per clip and config, in My Drive / hand2robot2 / results /**
- `<config>/<clip>/videos/hand_overlay.mp4` — the tracked hand on the clip, with how its distances were made
- `<config>/<clip>/videos/robot_sim.mp4` — the SO-101 in MuJoCo replaying it (red dot: where the hand was)
- `<config>/<clip>/videos/two_view.mp4` — both side by side
- `compare/<clip>.mp4` — A's two-view above B's
- `summary.csv` — the numbers; the hand and robot files of every run beside the videos

Run the cells **one by one**, top to bottom.

In [ ]:
#@title 1 · Settings
CODE_REPO = 'https://github.com/Aravind-0808/pipeline-colab-tmp.git'   # temporary, public; branch main
USE_DRIVE = True        # False in VS Code Colab (Drive cannot be mounted there): clips in /content/hand2robot2/clips
HANDS = 2               # 2 for a two-hand task (the cup handover), 1 for one hand
MAKE_VIDEOS = True
# WiLoR (config C): its checkpoints are old-style pickles from the warmshao/WiLoR-mini
# Hugging Face repo, which PyTorch refuses by default because a pickle can run code
# when loaded. True = you accept that, on this throw-away Colab runtime only.
ALLOW_WILOR_CHECKPOINTS = False

import os
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        ROOT = '/content/drive/MyDrive/hand2robot2'
    except Exception as e:
        print('Drive not available here (', e, ') — using /content/hand2robot2')
        ROOT = '/content/hand2robot2'
else:
    ROOT = '/content/hand2robot2'
os.makedirs(f'{ROOT}/clips', exist_ok=True)
print('ROOT =', ROOT, '| clips:', sorted(f for f in os.listdir(f'{ROOT}/clips') if f.endswith('.mp4')))

In [ ]:
#@title 2 · Code and packages (5–10 min the first time; re-run to pick up a newer push)
import os
if not os.path.isdir('/content/pipeline/.git'):
    !git clone -q {CODE_REPO} /content/pipeline
else:
    !git -C /content/pipeline pull -q
%cd /content/pipeline
!git log --oneline -1
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y ffmpeg libegl1 libgles2 > /dev/null 2>&1
# PyTorch pinned to the CUDA 12.6 build Colab's driver runs: a newer lerobot or
# torchcodec otherwise pulls PyTorch 2.11 for CUDA 13, whose cuSOLVER fails on
# Colab's driver (the depth model's lens estimate: CUSOLVER_STATUS_INTERNAL_ERROR).
# lerobot pinned too: without it pip walks back through releases that lack [dataset].
TORCH = ['torch==2.7.1', 'torchvision==0.22.1']
!pip install -q --index-url https://download.pytorch.org/whl/cu126 {' '.join(TORCH)}
!pip install -q -e ".[ml]" "lerobot[dataset]==0.6.1" mujoco {' '.join(TORCH)} --extra-index-url https://download.pytorch.org/whl/cu126
!python -c "import torch; print('PyTorch', torch.__version__, '| CUDA', torch.version.cuda); assert torch.__version__.startswith('2.7.1')"
!pip -q uninstall -y opencv-contrib-python opencv-python opencv-python-headless
!pip -q install "opencv-contrib-python-headless==5.0.0.93"
!sh scripts/install_perception.sh depth
!python scripts/download_retarget_weights.py --dir ./models
!python scripts/download_perception_weights.py --dir ./models depth
# The SO-101 for the MuJoCo videos: the official model, the same robot as models/so101.
if not os.path.isdir('/content/so101_sim/Simulation/SO101'):
    !git clone -q --filter=blob:none --no-checkout https://github.com/TheRobotStudio/SO-ARM100.git /content/so101_sim
    !cd /content/so101_sim && git sparse-checkout set --no-cone Simulation/SO101 && git checkout -q a758567c3978dfeefe282ede0500085a48fe8f78
!python -c "import cv2, mediapipe, scipy, moge, mujoco, dfpipeline.worker; print('imports OK')"

In [ ]:
#@title 3 · WiLoR (only when ALLOW_WILOR_CHECKPOINTS is True in cell 1)
# chumpy: a small stand-in, written below. The real chumpy no longer builds on
# Python 3.12+ / numpy 2, and WiLoR needs it only to read MANO's pickle (two classes).
import os, textwrap
if ALLOW_WILOR_CHECKPOINTS:
    d = '/content/chumpy_shim/chumpy'; os.makedirs(d, exist_ok=True)
    open(f'{d}/__init__.py', 'w').write('from chumpy.ch import Ch\n')
    open(f'{d}/ch.py', 'w').write(textwrap.dedent('''
        import numpy as np
        class Ch:
            # smplx reads .shape and slices before it converts with .r, so this
            # behaves as the array it holds.
            def __setstate__(self, state): self.__dict__.update(state)
            @property
            def r(self): return np.asarray(self.x)
            def __array__(self, dtype=None, copy=None):
                a = np.asarray(self.r)
                return a.astype(dtype) if dtype is not None else a
            @property
            def shape(self): return np.asarray(self.r).shape
            @property
            def ndim(self): return np.asarray(self.r).ndim
            @property
            def dtype(self): return np.asarray(self.r).dtype
            def __len__(self): return len(np.asarray(self.r))
            def __getitem__(self, i): return np.asarray(self.r)[i]
    '''))
    open(f'{d}/reordering.py', 'w').write(textwrap.dedent('''
        import numpy as np
        from chumpy.ch import Ch
        class Select(Ch):
            @property
            def r(self):
                v = np.asarray(self.a.r).ravel()[np.asarray(self.idxs)]
                s = getattr(self, "preferred_shape", None)
                return v.reshape(s) if s is not None else v
    '''))
    open('/content/chumpy_shim/pyproject.toml', 'w').write(textwrap.dedent('''
        [project]
        name = "chumpy"
        version = "0.0.0"
        dependencies = ["numpy"]
        [build-system]
        requires = ["setuptools>=61"]
        build-backend = "setuptools.build_meta"
        [tool.setuptools]
        packages = ["chumpy"]
    '''))
    !pip install -q --force-reinstall --no-deps /content/chumpy_shim
    !pip install -q --no-deps "wilor-mini @ git+https://github.com/warmshao/WiLoR-mini.git@ebec42f94c389070cdd7dda6fd1bf0b4a659c960" "smplx==0.1.28" "ultralytics==8.1.34"
    # --no-deps: thop and friends would otherwise upgrade PyTorch and pull a second
    # OpenCV, which overwrites the headless one (cv2 then has no VideoCapture).
    !pip install -q --no-deps timm einops roma py-cpuinfo thop
    !pip install -q scikit-image torch==2.7.1 torchvision==0.22.1 --extra-index-url https://download.pytorch.org/whl/cu126
    # Whatever came in, one OpenCV again: the headless contrib build MediaPipe takes.
    !pip -q uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
    !pip -q install --force-reinstall --no-deps "opencv-contrib-python-headless==5.0.0.93"
    !python -c "import cv2; assert hasattr(cv2, 'VideoCapture'); print('OpenCV OK', cv2.__version__)"
    !python scripts/download_perception_weights.py --dir ./models wilor
    os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
else:
    print('WiLoR skipped (ALLOW_WILOR_CHECKPOINTS is False): configs A and B only')

In [ ]:
#@title 4 · Do the models run here? (device, seconds a frame, GPU memory)
import os
os.environ.update({'PIPELINE_GPU': 'auto', 'RETARGET_MODELS_DIR': './models', 'GPU_MEMORY_FRACTION': '0.9',
                   'MUJOCO_GL': 'egl'})
!RETARGET_DEPTH_MODEL=moge-2-vitb python scripts/gpu_smoke.py
if ALLOW_WILOR_CHECKPOINTS:
    !RETARGET_HAND_MODEL=wilor python scripts/gpu_smoke.py

In [ ]:
#@title 5 · The runs (a few minutes a clip and config)
import csv, json, logging, shutil, sys, time, traceback
from pathlib import Path

logging.basicConfig(level=logging.WARNING)
from dfpipeline.config import settings
from dfpipeline.retarget import scene
from dfpipeline.retarget.pipeline import run_hand_stage, run_robot_stage
from dfpipeline.retarget import service as rs

CONFIGS = {
    'A_today': dict(retarget_hand_model='mediapipe', retarget_depth_model='off'),
    'B_depth': dict(retarget_hand_model='mediapipe', retarget_depth_model='moge-2-vitb'),
}
if ALLOW_WILOR_CHECKPOINTS:
    try:
        import wilor_mini  # noqa: F401
        CONFIGS['C_wilor_depth'] = dict(retarget_hand_model='wilor', retarget_depth_model='moge-2-vitb')
    except ImportError as e:
        print('WiLoR not installed, C skipped:', e)

clips = sorted(Path(ROOT, 'clips').glob('*.mp4'))
truth = {}
if Path(ROOT, 'truth.csv').exists():
    truth = {r['clip']: float(r['object_width_cm']) for r in csv.DictReader(open(Path(ROOT, 'truth.csv')))}
out_root = Path(ROOT, 'results'); out_root.mkdir(exist_ok=True)
print(len(clips), 'clips |', list(CONFIGS), '| hands:', HANDS)

def one(clip, name, cfg):
    for k, v in cfg.items():
        setattr(settings, k, v)
    scene.depth_model.cache_clear()
    out = out_root / name / clip.stem
    if out.exists():
        shutil.rmtree(out)
    cam_file = Path(str(clip) + '.camera.json')
    camera = json.loads(cam_file.read_text()) if cam_file.exists() else None
    t = time.time()
    code = run_hand_stage(clip, out / 'hand', hands=HANDS, camera=camera)
    row = {'clip': clip.name, 'config': name, 'hand_ok': code == 0, 'hand_s': round(time.time() - t, 1)}
    if code != 0:
        return row
    bundle = json.loads((out / 'hand' / 'hand_stage.json').read_text())
    row.update(rs.read_hand_metrics(out / 'hand'))
    row.update({f'p_{k}': v for k, v in (bundle.get('perception') or {}).items() if k != 'notes'})
    row['notes'] = '; '.join((bundle.get('perception') or {}).get('notes') or [])
    row['table_distance_m'] = bundle.get('table_distance_m')
    obj = bundle.get('object')
    row['object_width_cm'] = round(100 * obj['diameter_m'], 1) if obj else None
    row['truth_width_cm'] = truth.get(clip.name)
    t = time.time()
    rcode = run_robot_stage(out / 'hand', out / 'robot')
    row['robot_ok'] = rcode == 0
    row['robot_s'] = round(time.time() - t, 1)
    if rcode == 0:
        m = rs.read_metrics(out / 'robot')
        row.update({k: m.get(k) for k in ('reachable_fraction', 'position_error_median_m', 'slowdown')})
    return row

rows = []
for clip in clips:
    for name, cfg in CONFIGS.items():
        try:
            rows.append(one(clip, name, cfg))
        except Exception:
            rows.append({'clip': clip.name, 'config': name, 'error': traceback.format_exc()[-600:]})
        print(rows[-1].get('clip'), rows[-1].get('config'), rows[-1].get('hand_s'), rows[-1].get('error', ''))
json.dump(rows, open(out_root / 'rows.json', 'w'), indent=1, default=str)

In [ ]:
#@title 6 · The numbers
# After a restart, what cell 5 saved to Drive (results/rows.json) is read back:
# the runs are not repeated.
import json, os, traceback
from pathlib import Path
if 'rows' not in globals():
    out_root = Path(ROOT, 'results')
    rows = json.load(open(out_root / 'rows.json'))
    clips = sorted(Path(ROOT, 'clips').glob('*.mp4'))
    CONFIGS = {r['config']: None for r in rows}
    print('read back', len(rows), 'results:', list(CONFIGS))
import pandas as pd
df = pd.DataFrame(rows)
cols = ['clip', 'config', 'hand_ok', 'hand_s', 'p_lens_source', 'p_hfov_deg', 'p_table_source', 'table_distance_m',
        'p_hand_scale', 'p_hand_scale_measured', 'p_hand_scale_spread', 'p_table_up_angle_deg',
        'object_width_cm', 'truth_width_cm', 'hand_coverage', 'robot_ok', 'reachable_fraction',
        'position_error_median_m', 'slowdown', 'notes']
table = df[[c for c in cols if c in df.columns]]
if 'truth_width_cm' in table and table['truth_width_cm'].notna().any():
    table = table.assign(width_error_cm=(table['object_width_cm'] - table['truth_width_cm']).abs())
    print(table.groupby('config')['width_error_cm'].median().rename('median object-width error, cm'))
print(table.groupby('config')[['hand_s', 'reachable_fraction', 'position_error_median_m', 'slowdown']].median())
table.to_csv(out_root / 'summary.csv', index=False)
table

In [ ]:
#@title 7 · The videos (to Drive: results/<config>/<clip>/videos/, results/compare/)
# After a restart, what cell 5 saved to Drive (results/rows.json) is read back:
# the runs are not repeated.
import json, os, traceback
from pathlib import Path
if 'rows' not in globals():
    out_root = Path(ROOT, 'results')
    rows = json.load(open(out_root / 'rows.json'))
    clips = sorted(Path(ROOT, 'clips').glob('*.mp4'))
    CONFIGS = {r['config']: None for r in rows}
    print('read back', len(rows), 'results:', list(CONFIGS))
import sys
sys.path.insert(0, '/content/pipeline/research')
os.environ['MUJOCO_GL'] = 'egl'
import visualize
SO101 = Path('/content/so101_sim/Simulation/SO101')
if MAKE_VIDEOS:
    (out_root / 'compare').mkdir(exist_ok=True)
    for clip in clips:
        made = {}
        for name in CONFIGS:
            run = out_root / name / clip.stem
            if not (run / 'hand' / 'hand_stage.json').exists():
                continue
            try:
                made[name] = visualize.render(clip, run, f'{name}  {clip.name}', so101_dir=SO101)
                print(clip.name, name, {k: v.name for k, v in made[name].items()})
            except Exception:
                print(clip.name, name, 'video failed:', traceback.format_exc()[-400:])
        if 'two_view' in made.get('A_today', {}) and 'two_view' in made.get('B_depth', {}):
            dest = visualize.compare(made['A_today']['two_view'], made['B_depth']['two_view'],
                                     out_root / 'compare' / f'{clip.stem}.mp4')
            print('A over B:', dest)

In [ ]:
#@title 8 · Watch one here (the first clip's A-over-B comparison)
from IPython.display import Video
first = sorted((out_root / 'compare').glob('*.mp4'))
Video(str(first[0]), embed=True, width=960) if first else print('no comparison video yet')

**Send back** `results/summary.csv` and tell me what the videos show — does the robot in **B** follow the hand better than in **A**? When you are done: **Runtime → Disconnect and delete runtime**.